In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.impute import SimpleImputer

df = pd.read_csv(
    "listings.csv.gz",
    compression="gzip"
)

print("Dimensions du fichier brut :", df.shape)

display(df.head())

Dimensions du fichier brut : (9647, 79)


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,56766,https://www.airbnb.com/rooms/56766,20250918041715,2025-09-18,city scrape,"Duplex avec vue, centre ville",Beautifull duplex terrace on Lyon's roofs ! Co...,"the general atmosphere, the magnificent archit...",https://a0.muscache.com/pictures/hosting/Hosti...,269557,...,4.88,4.86,4.61,"Available with a mobility lease only (""bail mo...",f,1,1,0,0,0.31
1,77104,https://www.airbnb.com/rooms/77104,20250918041715,2025-09-18,city scrape,Nice Flat in the center of Lyon,NaN,NaN,https://a0.muscache.com/pictures/518769/186679...,411285,...,5.00,4.88,4.85,6938113058353,f,1,1,0,0,0.15
2,134539,https://www.airbnb.com/rooms/134539,20250918041715,2025-09-18,city scrape,"Centre of old town,home sweet home",Ideal situation and cosy..In the center of the...,It's more than charming!<br />It's the most be...,https://a0.muscache.com/pictures/hosting/Hosti...,660135,...,4.62,4.86,4.44,6938513221658,f,1,1,0,0,0.77
3,137132,https://www.airbnb.com/rooms/137132,20250918041715,2025-09-18,city scrape,"Bel apartment, 2 rooms, Lyon center","- Nice 2 rooms apartment, on the 6th and last ...",Four very nice: residential but with lots of r...,https://a0.muscache.com/pictures/1085278/6420e...,670404,...,4.84,4.95,4.53,0693861800177,f,1,1,0,0,0.12
4,174304,https://www.airbnb.com/rooms/174304,20250918041715,2025-09-18,city scrape,Quartier Terreaux 4/6 pers. Hyper centre / Clim,"In a typical Lyonnais building, this beautiful...",A prime location in the center of the city of ...,https://a0.muscache.com/pictures/2c68bbca-08dd...,832917,...,4.86,4.88,4.56,6938112368465,f,2,2,0,0,2.90


### nettoyage 

In [14]:
def charger_airbnb(url_csv):
    """
    Charge le fichier Airbnb, sélectionne les colonnes numériques utiles
    et traite les valeurs manquantes de manière adaptée.

    Retourne un DataFrame numérique sans valeur manquante.
    """

    df = pd.read_csv(url_csv)

    colonnes_utiles = [
        "price",
        "minimum_nights",
        "number_of_reviews",
        "availability_365",
        "accommodates",
        "reviews_per_month"
    ]

    airbnb = df[colonnes_utiles].copy()

    # Nettoyage de la colonne price
    airbnb["price"] = (
        airbnb["price"]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace("€", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )

    # Conversion de toutes les colonnes en nombres
    for colonne in colonnes_utiles:
        airbnb[colonne] = pd.to_numeric(
            airbnb[colonne],
            errors="coerce"
        )

    print("Valeurs manquantes avant traitement :")
    print(airbnb.isna().sum())

    nombre_initial = len(airbnb)

    # Le prix est essentiel : les annonces sans prix sont supprimées
    airbnb = airbnb.dropna(subset=["price"])

    nombre_sans_prix = nombre_initial - len(airbnb)

    # Si une annonce n'a aucun avis, reviews_per_month prend la valeur 0
    condition_sans_avis = (
        airbnb["reviews_per_month"].isna()
        & (airbnb["number_of_reviews"] == 0)
    )

    airbnb.loc[
        condition_sans_avis,
        "reviews_per_month"
    ] = 0

    # Les autres valeurs manquantes sont remplacées par la médiane
    imputer = SimpleImputer(strategy="median")

    airbnb[colonnes_utiles] = imputer.fit_transform(
        airbnb[colonnes_utiles]
    )

    airbnb = airbnb.reset_index(drop=True)

    print("\nTraitement effectué :")
    print("Annonces supprimées car le prix était manquant :", nombre_sans_prix)
    print(
        "Valeurs reviews_per_month remplacées par 0 :",
        condition_sans_avis.sum()
    )

    print("\nValeurs manquantes après traitement :")
    print(airbnb.isna().sum())

    print(
        f"\nListings chargés : {len(airbnb)} lignes, "
        f"{airbnb.shape[1]} colonnes numériques retenues"
    )

    return airbnb

In [15]:
airbnb = charger_airbnb("listings.csv.gz")

print("\nDimensions du DataFrame retourné :")
print(airbnb.shape)

print("\nPremières annonces après traitement :")
display(airbnb.head())

Valeurs manquantes avant traitement :
price                4391
minimum_nights          0
number_of_reviews       0
availability_365        0
accommodates            0
reviews_per_month    1823
dtype: int64

Traitement effectué :
Annonces supprimées car le prix était manquant : 4391
Valeurs reviews_per_month remplacées par 0 : 629

Valeurs manquantes après traitement :
price                0
minimum_nights       0
number_of_reviews    0
availability_365     0
accommodates         0
reviews_per_month    0
dtype: int64

Listings chargés : 5256 lignes, 6 colonnes numériques retenues

Dimensions du DataFrame retourné :
(5256, 6)

Premières annonces après traitement :


,price,minimum_nights,number_of_reviews,availability_365,accommodates,reviews_per_month
0,118.0,30.0,56.0,91.0,5.0,0.31
1,90.0,2.0,27.0,83.0,4.0,0.15
2,77.0,2.0,134.0,80.0,2.0,0.77
3,88.0,365.0,20.0,364.0,2.0,0.12
4,207.0,2.0,465.0,224.0,6.0,2.90


### Standardisation

In [18]:
scaler_airbnb = StandardScaler()

print(scaler_airbnb)
X_scaled = scaler_airbnb.fit_transform(airbnb)


StandardScaler()


### Fonction choisir_k

In [22]:
def choisir_k(X_scaled, k_range=range(2, 9)):
    

    resultats = []

    for k in k_range:

        modele = KMeans(
            n_clusters=k,
            n_init=10,
            random_state=42
        )

        labels = modele.fit_predict(X_scaled)

        inertie = modele.inertia_

        silhouette = silhouette_score(
            X_scaled,
            labels,
            sample_size=min(3000, len(X_scaled)),
            random_state=42
        )

        resultats.append({
            "k": k,
            "inertie": inertie,
            "silhouette": silhouette
        })

        print(
            f"k={k} : "
            f"inertie={inertie:.0f} "
            f"silhouette={silhouette:.2f}"
        )

    tableau_resultats = pd.DataFrame(resultats)

    return tableau_resultats

In [23]:
resultats_k = choisir_k(
    X_scaled,
    k_range=range(2, 9)
)

display(
    resultats_k.round({
        "inertie": 0,
        "silhouette": 3
    })
)

k=2 : inertie=25903 silhouette=0.32
k=3 : inertie=20785 silhouette=0.37
k=4 : inertie=16831 silhouette=0.35
k=5 : inertie=13451 silhouette=0.31
k=6 : inertie=11910 silhouette=0.31
k=7 : inertie=10555 silhouette=0.30
k=8 : inertie=9662 silhouette=0.29


,k,inertie,silhouette
0,2,25903.0,0.324
1,3,20785.0,0.368
2,4,16831.0,0.349
3,5,13451.0,0.306
4,6,11910.0,0.305
5,7,10555.0,0.304
6,8,9662.0,0.285


### Sélection du meilleur k

In [25]:
meilleure_ligne = resultats_k.loc[
    resultats_k["silhouette"].idxmax()
]

k_retenu = int(meilleure_ligne["k"])

print("Segment retenu : k =", k_retenu)

print(
    "Score de silhouette correspondant :",
    round(meilleure_ligne["silhouette"], 3)
)

Segment retenu : k = 3
Score de silhouette correspondant : 0.368


### les cas demandés 

#### Cas normal

In [27]:
kmeans_normal = KMeans(
    n_clusters=k_retenu,
    n_init=10,
    random_state=42
)
labels_normaux = kmeans_normal.fit_predict(X_scaled)

airbnb_clusters = airbnb.copy()
airbnb_clusters["cluster"] = labels_normaux

print("Nombre d'annonces dans chaque cluster :")

print(
    airbnb_clusters["cluster"]
    .value_counts()
    .sort_index()
)

Nombre d'annonces dans chaque cluster :
cluster
0     843
1    4328
2      85
Name: count, dtype: int64


##### Profil moyen de chaque segment

In [31]:
profils_clusters = (
    airbnb_clusters
    .groupby("cluster")
    .agg(
        nombre_annonces=("cluster", "size"),
        prix_moyen=("price", "mean"),
        nuits_minimales=("minimum_nights", "mean"),
        nombre_avis=("number_of_reviews", "mean"),
        disponibilite=("availability_365", "mean"),
        capacite=("accommodates", "mean")
    )
)

print("Profil moyen de chaque segment :")

display(profils_clusters.round(2))

Profil moyen de chaque segment :


,nombre_annonces,prix_moyen,nuits_minimales,nombre_avis,disponibilite,capacite
cluster,,,,,,
0,843,84.34,1.74,258.47,211.60,3.04
1,4328,107.42,4.94,32.77,193.00,3.26
2,85,130.47,364.24,23.22,333.68,3.71


##### Fonction de description des segments

In [33]:
def classer_clusters_par_niveau(serie):
    """
    Classe les clusters du niveau le plus faible
    au niveau le plus élevé.
    """

    nombre_clusters = len(serie)
    ordre = serie.sort_values().index

    if nombre_clusters == 3:
        niveaux = ["faible", "moyen", "élevé"]

    elif nombre_clusters == 4:
        niveaux = ["très faible", "faible", "élevé", "très élevé"]

    else:
        niveaux = [
            f"niveau {i + 1}"
            for i in range(nombre_clusters)
        ]

    classement = {}

    for cluster, niveau in zip(ordre, niveaux):
        classement[cluster] = niveau

    return classement
niveaux_prix = classer_clusters_par_niveau(
    profils_clusters["prix_moyen"]
)

niveaux_avis = classer_clusters_par_niveau(
    profils_clusters["nombre_avis"]
)

niveaux_disponibilite = classer_clusters_par_niveau(
    profils_clusters["disponibilite"]
)

niveaux_capacite = classer_clusters_par_niveau(
    profils_clusters["capacite"]
)

print("Classement relatif des prix :")
print(niveaux_prix)

print("\nClassement relatif du nombre d'avis :")
print(niveaux_avis)

print("\nClassement relatif de la disponibilité :")
print(niveaux_disponibilite)

print("\nClassement relatif de la capacité :")
print(niveaux_capacite)

Classement relatif des prix :
{0: 'faible', 1: 'moyen', 2: 'élevé'}

Classement relatif du nombre d'avis :
{2: 'faible', 1: 'moyen', 0: 'élevé'}

Classement relatif de la disponibilité :
{1: 'faible', 0: 'moyen', 2: 'élevé'}

Classement relatif de la capacité :
{0: 'faible', 1: 'moyen', 2: 'élevé'}


In [34]:
print("Description finale des segments :\n")

print(
    "Cluster 0 : petits budgets, beaucoup d'avis, "
    "disponibilité moyenne et logements plutôt adaptés "
    "aux voyageurs seuls ou aux couples."
)

print(
    "Cluster 1 : prix intermédiaires, nombre d'avis moyen, "
    "faible disponibilité et logements adaptés aux petits groupes."
)

print(
    "Cluster 2 : logements premium, peu d'avis, "
    "forte disponibilité et logements plutôt adaptés "
    "aux familles ou aux groupes."
)

Description finale des segments :

Cluster 0 : petits budgets, beaucoup d'avis, disponibilité moyenne et logements plutôt adaptés aux voyageurs seuls ou aux couples.
Cluster 1 : prix intermédiaires, nombre d'avis moyen, faible disponibilité et logements adaptés aux petits groupes.
Cluster 2 : logements premium, peu d'avis, forte disponibilité et logements plutôt adaptés aux familles ou aux groupes.


#### cas limite 

In [35]:
dispersion_variables = (
    airbnb.std()
    .sort_values(ascending=False)
    .to_frame("Écart-type")
)

print("Dispersion des variables avant standardisation :")

display(dispersion_variables.round(2))

Dispersion des variables avant standardisation :


,Écart-type
availability_365,121.39
number_of_reviews,118.42
price,101.24
minimum_nights,46.38
reviews_per_month,2.01
accommodates,1.83


#### créer le modèle sans standardisation

In [37]:
kmeans_sans_standardisation = KMeans(
    n_clusters=k_retenu,
    n_init=10,
    random_state=42
)

labels_sans_standardisation = (
    kmeans_sans_standardisation.fit_predict(airbnb)
)

print("Nombre d'annonces dans chaque cluster :")

print(
    pd.Series(labels_sans_standardisation)
    .value_counts()
    .sort_index()
)

Nombre d'annonces dans chaque cluster :
0    2267
1    2508
2     481
Name: count, dtype: int64


#### ajouter les clusters

In [38]:
airbnb_sans_standardisation = airbnb.copy()

airbnb_sans_standardisation["cluster"] = (
    labels_sans_standardisation
)

print("Premières annonces avec leur cluster :")

display(airbnb_sans_standardisation.head())

Premières annonces avec leur cluster :


,price,minimum_nights,number_of_reviews,availability_365,accommodates,reviews_per_month,cluster
0,118.0,30.0,56.0,91.0,5.0,0.31,0
1,90.0,2.0,27.0,83.0,4.0,0.15,0
2,77.0,2.0,134.0,80.0,2.0,0.77,0
3,88.0,365.0,20.0,364.0,2.0,0.12,1
4,207.0,2.0,465.0,224.0,6.0,2.90,2


In [39]:
profils_sans_standardisation = (
    airbnb_sans_standardisation
    .groupby("cluster")
    .agg(
        nombre_annonces=("cluster", "size"),
        prix_moyen=("price", "mean"),
        nuits_minimales=("minimum_nights", "mean"),
        nombre_avis=("number_of_reviews", "mean"),
        disponibilite=("availability_365", "mean"),
        capacite=("accommodates", "mean")
    )
)

print("Profils obtenus sans standardisation :")

display(profils_sans_standardisation.round(2))

Profils obtenus sans standardisation :


,nombre_annonces,prix_moyen,nuits_minimales,nombre_avis,disponibilite,capacite
cluster,,,,,,
0,2267,95.72,4.30,40.19,79.27,3.17
1,2508,115.76,16.89,37.03,302.95,3.33
2,481,82.71,3.50,369.44,213.15,3.03


#### Cas adversarial

In [41]:
annonce_aberrante = airbnb.median().to_frame().T

annonce_aberrante["price"] = 100_000
display(annonce_aberrante)
airbnb_adversarial = pd.concat(
    [airbnb, annonce_aberrante],
    ignore_index=True
)

print("Dimensions avant l'ajout :", airbnb.shape)
print("Dimensions après l'ajout :", airbnb_adversarial.shape)

,price,minimum_nights,number_of_reviews,availability_365,accommodates,reviews_per_month
0,100000,2.0,22.0,219.5,2.0,1.0


Dimensions avant l'ajout : (5256, 6)
Dimensions après l'ajout : (5257, 6)


#### vérifier le prix maximal

In [42]:
print(
    "Prix maximal avant l'ajout :",
    airbnb["price"].max(),
    "euros"
)

print(
    "Prix maximal après l'ajout :",
    airbnb_adversarial["price"].max(),
    "euros"
)

Prix maximal avant l'ajout : 2281.0 euros
Prix maximal après l'ajout : 100000.0 euros


#### standardiser le nouveau dataset et création du model 

In [44]:
scaler_adversarial = StandardScaler()

X_adversarial_scaled = scaler_adversarial.fit_transform(
    airbnb_adversarial
)

kmeans_adversarial = KMeans(
    n_clusters=k_retenu,
    n_init=10,
    random_state=42
)

labels_adversariaux = kmeans_adversarial.fit_predict(
    X_adversarial_scaled
)

print("Nombre d'annonces dans chaque cluster :")

print(
    pd.Series(labels_adversariaux)
    .value_counts()
    .sort_index()
)

Nombre d'annonces dans chaque cluster :
0    4358
1     898
2       1
Name: count, dtype: int64


#### identifier le cluster de l’anomalie

In [45]:
cluster_annonce_aberrante = labels_adversariaux[-1]

print(
    "Cluster attribué à l'annonce à 100 000 euros :",
    cluster_annonce_aberrante
)

Cluster attribué à l'annonce à 100 000 euros : 2


#### centres avant l’anomalie

In [46]:
centres_normaux = pd.DataFrame(
    scaler_airbnb.inverse_transform(
        kmeans_normal.cluster_centers_
    ),
    columns=airbnb.columns
)

centres_normaux = centres_normaux.sort_values(
    by="price"
).reset_index(drop=True)

print("Centres des clusters avant l'anomalie :")

display(centres_normaux.round(2))

Centres des clusters avant l'anomalie :


,price,minimum_nights,number_of_reviews,availability_365,accommodates,reviews_per_month
0,84.40,1.74,258.95,212.00,3.04,5.16
1,107.40,4.94,32.78,192.93,3.26,1.07
2,130.47,364.24,23.22,333.68,3.71,0.25


#### centres après l’anomalie

In [47]:
centres_adversariaux = pd.DataFrame(
    scaler_adversarial.inverse_transform(
        kmeans_adversarial.cluster_centers_
    ),
    columns=airbnb.columns
)

centres_adversariaux = centres_adversariaux.sort_values(
    by="price"
).reset_index(drop=True)

print("Centres des clusters après l'anomalie :")

display(centres_adversariaux.round(2))

Centres des clusters après l'anomalie :


,price,minimum_nights,number_of_reviews,availability_365,accommodates,reviews_per_month
0,86.71,2.54,250.14,212.52,3.10,5.06
1,107.68,11.82,31.35,195.31,3.26,1.02
2,100000.00,2.00,22.00,219.50,2.00,1.00


#### comparer les centres

In [48]:
prix_centre_avant = centres_normaux["price"].max()
prix_centre_apres = centres_adversariaux["price"].max()

print(
    "Prix maximal d'un centre avant l'anomalie :",
    round(prix_centre_avant, 2),
    "euros"
)

print(
    "Prix maximal d'un centre après l'anomalie :",
    round(prix_centre_apres, 2),
    "euros"
)

Prix maximal d'un centre avant l'anomalie : 130.47 euros
Prix maximal d'un centre après l'anomalie : 100000.0 euros


#### mesurer le déplacement

In [49]:
deplacement_centre = prix_centre_apres - prix_centre_avant

print(
    "Déplacement du centre le plus cher :",
    round(deplacement_centre, 2),
    "euros"
)

Déplacement du centre le plus cher : 99869.53 euros
